In [87]:
import pandas as pd
import re

In [88]:
kagglepath =  "../../dataset/kgdataset.csv"
surveypath = "../../dataset/survey_data.xlsx"
df = pd.read_csv(kagglepath)
df2 = pd.read_excel(surveypath)
df2.drop(df2.columns[0],axis=1, inplace=True)
df2.columns = df.columns
combined = pd.concat([df, df2], ignore_index=True)
combined


,Age,Sleep Hours,Feel Rested,Daily Screen Time,Use Before Sleep,Stress Level,Anxiety/Low Mood,Wellness Apps,Sleep Quality,Screen Time Affects Sleep?
0,47,4,Yes,11,Yes,10,No,No,Bad,Yes
1,48,4,No,8,No,7,Yes,Yes,Bad,Yes
2,40,7,Sometimes,10,No,4,No,Yes,Bad,Yes
3,26,10,Yes,5,Yes,1,Yes,No,Good,No
4,47,5,Yes,7,Yes,7,Yes,No,Good,Yes
...,...,...,...,...,...,...,...,...,...,...
1181,21,10,สดชื่น,8,ใช่,4,ไม่หงุดหงิด/ไม่วิตกกังวล,ใช่,ดี,ไม่แน่ใจ
1182,17,7-8,ไม่สดชื่น,>8ชม.,ใช่,8,ไม่หงุดหงิด/ไม่วิตกกังวล,ไม่ใช่,ดี,ไม่แน่ใจ
1183,17,6-8,สดชื่น,3-4,ใช่,6,ไม่หงุดหงิด/ไม่วิตกกังวล,ใช่,ไม่ดี,ใช่
1184,17,6,ไม่สดชื่น,4,ใช่,6,ไม่หงุดหงิด/ไม่วิตกกังวล,ไม่ใช่,ไม่ดี,ใช่


In [89]:
dfclean = pd.DataFrame(combined)
dfclean["Feel Rested"] = dfclean["Feel Rested"].replace("ไม่สดชื่น", "No").replace("สดชื่น", "Yes")
dfclean["Use Before Sleep"] = dfclean["Use Before Sleep"].replace("ไม่ใช่", "No").replace("ใช่", "Yes")
dfclean["Anxiety/Low Mood"] = dfclean["Anxiety/Low Mood"].replace("ไม่หงุดหงิด/ไม่วิตกกังวล", "No").replace("หงุดหงิด/วิตกกังวล", "Yes")
dfclean["Wellness Apps"] = dfclean["Wellness Apps"].replace("ไม่ใช่", "No").replace("ใช่", "Yes")
dfclean["Sleep Quality"] = dfclean["Sleep Quality"].replace("ไม่ดี", "Bad").replace("ดี", "Good")
dfclean["Screen Time Affects Sleep?"] = dfclean["Screen Time Affects Sleep?"].replace("ไม่แน่ใจ", "Not Sure").replace("ใช่", "Yes").replace("ไม่ใช่", "No")


In [90]:
#check for null values and duplicates
print(dfclean.isnull().sum())
duplicates = dfclean[dfclean.duplicated()]
print("Duplicate Rows:")
print(duplicates)
dfclean = dfclean.drop_duplicates()
print("DataFrame after removing duplicates:")
print(dfclean)

Age                           0
Sleep Hours                   0
Feel Rested                   0
Daily Screen Time             0
Use Before Sleep              0
Stress Level                  0
Anxiety/Low Mood              0
Wellness Apps                 0
Sleep Quality                 0
Screen Time Affects Sleep?    0
dtype: int64
Duplicate Rows:
    Age Sleep Hours Feel Rested Daily Screen Time Use Before Sleep  \
181  15          10          No                 2               No   
358  53           4   Sometimes                 9              Yes   
452  17           8          No                 4              Yes   
739  39           8   Sometimes                 2               No   
955  51          10   Sometimes                 5               No   

     Stress Level Anxiety/Low Mood Wellness Apps Sleep Quality  \
181             5               No            No          Good   
358             8               No            No           Bad   
452             3               

In [91]:
#clean age
def extract_number(value):
    if "-" in str(value):
        value = re.sub(r'[^\d\-\.]', '', value)
        values = str(value).split("-")
        return f"{values[0]}-{values[1]}"
    elif isinstance(value, str):
        result = pd.Series(value).str.extract('(\d+)') 
        return result[0].iloc[0] if not result.empty else None

    else:
        return str(value)      
dfclean["Age"] = dfclean["Age"].apply(extract_number)

#clean sleep hours by mean
def clean_range_with_mean(value):
    if isinstance(value, str):
        if '-' in value:
            try:
                low, high = map(float, value .split('-'))
                return (low + high) / 2
            except ValueError:
                return value
        else:
            try:
                return float(value)
            except ValueError:
                return value
    else:
        return value
dfclean["Sleep Hours"] = dfclean["Sleep Hours"].apply(extract_number)
dfclean["Sleep Hours"] = dfclean["Sleep Hours"].apply(clean_range_with_mean)

#Clean Daily Screen Time
dfclean["Daily Screen Time"] = dfclean["Daily Screen Time"].apply(extract_number)
dfclean["Daily Screen Time"] = dfclean["Daily Screen Time"].apply(clean_range_with_mean)
print(dfclean["Sleep Hours"].isnull().sum())
print(dfclean["Age"].isnull().sum())
print(dfclean["Daily Screen Time"].isnull().sum())

0
0
0


In [92]:
dfclean.tail(60)

,Age,Sleep Hours,Feel Rested,Daily Screen Time,Use Before Sleep,Stress Level,Anxiety/Low Mood,Wellness Apps,Sleep Quality,Screen Time Affects Sleep?
1126,20,8.0,No,10.0,Yes,1,No,Yes,Bad,Yes
1127,22,5.0,No,2.0,Yes,4,No,No,Bad,Yes
1128,19,6.0,Yes,10.0,Yes,7,Yes,No,Good,Yes
1129,20,8.0,Yes,4.0,Yes,7,No,No,Good,Yes
1130,23,9.0,No,10.0,Yes,7,Yes,Yes,Good,Not Sure
1131,100,1.0,No,20.0,Yes,10,Yes,Yes,Bad,Yes
1132,20,6.0,Yes,11.0,Yes,7,No,Yes,Bad,Yes
1133,19,5.0,No,10.0,Yes,7,Yes,No,Good,Yes
1134,21,10.0,No,8.0,Yes,7,Yes,No,Bad,Yes
1135,21,7.0,Yes,6.0,Yes,3,No,No,Good,Yes
